# Chapter 5 &mdash; Best Practices: Mnemonic State Names

**Concept 3 of the Chapter 5 decomposition:** *Best Practices: Mnemonic State Names and Documented Transitions*

Name states for what they remember; keep transitions consistent with the names; comment every line.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Mnemonic-State-Names/Concept-Mnemonic-State-Names.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Two habits separate DFA you can debug from DFA you cannot:

* **name a state for what it remembers**, not `q0, q1, q2`;
* **write a `!!` comment on every transition**, saying why it goes there.

Jove's markdown supports both. `md2mc` reads the leading letters (`I`, `F`, `IF`) for
the machine's structure and ignores the rest of the name, so `IF_even_even` is both
initial and final **and** self-documenting.

The pay-off comes when a test fails: a mnemonic name tells you instantly which
invariant broke.

## 2. Definitions

### The same machine, opaque

In [ ]:
opaque = md2mc('''DFA
IF : 0 -> A
IF : 1 -> B
A  : 0 -> IF
A  : 1 -> C
B  : 0 -> C
B  : 1 -> IF
C  : 0 -> B
C  : 1 -> A
''')

### and mnemonic, with a comment per transition

In [ ]:
clear = md2mc('''DFA
!!  State name records (parity of 0s, parity of 1s)
IF_ev0_ev1 : 0 -> S_od0_ev1   !! saw a 0: zero-parity flips
IF_ev0_ev1 : 1 -> S_ev0_od1   !! saw a 1: one-parity flips
S_od0_ev1  : 0 -> IF_ev0_ev1  !! second 0 restores even
S_od0_ev1  : 1 -> S_od0_od1
S_ev0_od1  : 0 -> S_od0_od1
S_ev0_od1  : 1 -> IF_ev0_ev1  !! second 1 restores even
S_od0_od1  : 0 -> S_ev0_od1
S_od0_od1  : 1 -> S_od0_ev1
''')

<!-- nav-strip -->

---

&larr;&nbsp;[Ch5&nbsp;2.&nbsp;Cross-Checking a Specification: the Equal-Changes Language $L_{eqc}$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Cross-Checking-Leqc/Concept-Cross-Checking-Leqc.ipynb) &nbsp;&middot;&nbsp; [**Chapter 5** index](https://github.com/ganeshutah/Jove/blob/master/Chapter5/README.md) &nbsp;&middot;&nbsp; [Ch5&nbsp;4.&nbsp;Verification by Construction Twice: Minimal DFA Uniqueness and `iso_dfa`](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Verification-By-Construction-Twice/Concept-Verification-By-Construction-Twice.ipynb)&nbsp;&rarr;

---

## 3. Tests

Same language &mdash; the names cost nothing at runtime.

In [ ]:
print("same language? ", langeq_dfa(opaque, clear))
print("isomorphic?    ", iso_dfa(opaque, clear))
assert langeq_dfa(opaque, clear) and iso_dfa(opaque, clear)

But the names are **checkable invariants**: the state must match the parities seen.

In [ ]:
def parity_pair(s): return (s.count('0') % 2, s.count('1') % 2)
tag = {'IF_ev0_ev1': (0,0), 'S_od0_ev1': (1,0), 'S_ev0_od1': (0,1), 'S_od0_od1': (1,1)}

from itertools import product
for k in range(6):
    for p in product('01', repeat=k):
        s = ''.join(p)
        assert tag[run_dfa(clear, s)] == parity_pair(s), s
print("every state's NAME matches what it actually remembers, on all strings up to length 5")
print("\nThat assertion is only writable because the names mean something.")

With opaque names the same check is impossible to state, so the bug would hide.

In [ ]:
print("states of the opaque machine :", sorted(opaque["Q"]))
print("what does 'C' remember? -- you have to re-derive it every time.")

## 4. Animation

Mnemonic names make the picture readable too.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(clear, FuseEdges=True)

## 5. Exercises


1. Rename the states of a DFA you wrote earlier. Did you find a bug?
2. Write the invariant assertion for the `01`-containing DFA of Chapter 4.
3. Why can `md2mc` ignore everything after the leading `I`/`F`?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter5/Concept-Mnemonic-State-Names')